# RAG Support Chatbot — Milestone 3 (VS Code / local version)
## Advanced Techniques & Deployment

This notebook **tests** the RAG chain interactively.
The actual production code lives in:
- `src/rag_chain.py` — core RAG logic (retrieval + generation)
- `src/api.py`       — FastAPI REST server wrapping the chain

**Pipeline per query:**
```
user question
  → embed with sentence-transformers (all-MiniLM-L6-v2)
  → hybrid retrieve top-3 from FAISS index
  → build prompt: system message + context docs + question
  → generate answer with flan-t5-base (local CPU)
  → return answer + sources
```

**Steps:**
```
Step 1 → Install new dependencies
Step 2 → Load all models (embedding + FAISS + LLM)
Step 3 → Test the RAG chain interactively
Step 4 → Evaluate answer quality
Step 5 → Test the REST API
Step 6 → Security notes
```

## Step 1 — Install new dependencies

In [ ]:
# Run this once in your activated venv terminal:
#   pip install transformers torch fastapi uvicorn[standard] pydantic httpx
#
# torch is needed by transformers for flan-t5 inference on CPU.
# httpx is needed to test the API from inside the notebook.
#
# Uncomment to install from inside the notebook:
# %pip install transformers torch fastapi uvicorn[standard] pydantic httpx

## Step 2 — Load all models

In [ ]:
import sys
import os

# Make sure Python can find the src/ package
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.rag_chain import load_all, ask, search

# Load everything into memory.
# First run: downloads flan-t5-base (~250MB) from Hugging Face.
# Subsequent runs: loads from local cache — much faster.
load_all()
print('All models loaded and ready.')

## Step 3 — Test the RAG chain interactively

In [ ]:
def print_result(result: dict) -> None:
    """Pretty-print a RAG chain result dict."""
    print(f"QUERY    : {result['query']}")
    print(f"RETRIEVAL: {result['retrieval']}")
    print(f"\nANSWER:\n{result['answer']}")
    print(f"\nSOURCES USED ({len(result['sources'])} docs):")
    for i, src in enumerate(result['sources'], 1):
        print(f"  #{i} [{src['category']} -> {src['intent']}]  score={src['score']:.4f}")
        print(f"      Q: {src['instruction']}")
        print(f"      A: {src['response'][:100]}...")
    print('='*65)

In [ ]:
# --- Test 1: Order cancellation ---
result = ask("I want to cancel my order")
print_result(result)

In [ ]:
# --- Test 2: Missing package ---
result = ask("My package hasn't arrived and it's been 2 weeks")
print_result(result)

In [ ]:
# --- Test 3: Password reset ---
result = ask("I forgot my password and I cannot log into my account")
print_result(result)

In [ ]:
# --- Test 4: Double charge ---
result = ask("I was charged twice for the same order")
print_result(result)

In [ ]:
# --- Test 5: Defective product return ---
result = ask("The product I received is broken, how do I return it?")
print_result(result)

In [ ]:
# --- Test 6: Noisy/informal query (simulates real customer typing) ---
result = ask("whr is my ordr?? its been forever")
print_result(result)

## Step 4 — Evaluate answer quality

We compare the LLM-generated answer against the gold response from the dataset.
A high ROUGE score means the generated answer closely mirrors the expected response.

In [ ]:
import pandas as pd
import numpy as np
import nltk
from tqdm.auto import tqdm
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

rouge  = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
smooth = SmoothingFunction().method1

test_df = pd.read_csv('../data/test_df.csv')

EVAL_SAMPLE = 50   # keep small — each row calls the LLM which is slow on CPU
sample = test_df.sample(EVAL_SAMPLE, random_state=42).reset_index(drop=True)

bleu_scores, r1, r2, rl = [], [], [], []

print(f'Evaluating {EVAL_SAMPLE} test queries end-to-end (retrieve + generate)...')
print('This takes a few minutes on CPU — each query runs the full LLM pipeline.\n')

for _, row in tqdm(sample.iterrows(), total=EVAL_SAMPLE):
    result      = ask(row['instruction_clean'], top_k=3)
    gold        = str(row['response_clean'])
    generated   = result['answer']

    # BLEU
    ref = nltk.word_tokenize(gold.lower())
    hyp = nltk.word_tokenize(generated.lower())
    bleu_scores.append(sentence_bleu([ref], hyp, smoothing_function=smooth))

    # ROUGE
    rs = rouge.score(gold, generated)
    r1.append(rs['rouge1'].fmeasure)
    r2.append(rs['rouge2'].fmeasure)
    rl.append(rs['rougeL'].fmeasure)

print('\n' + '='*50)
print('END-TO-END EVALUATION RESULTS (RAG chain)')
print('='*50)
print(f'  Sample size : {EVAL_SAMPLE}')
print(f'  Avg BLEU    : {np.mean(bleu_scores):.4f}')
print(f'  Avg ROUGE-1 : {np.mean(r1):.4f}')
print(f'  Avg ROUGE-2 : {np.mean(r2):.4f}')
print(f'  Avg ROUGE-L : {np.mean(rl):.4f}')
print()
print('Score guide for this task:')
print('  ROUGE-1 > 0.40 = good retrieval   ROUGE-L > 0.35 = coherent generation')

## Step 5 — Test the REST API

**Before running this step**, start the API server in a separate VS Code terminal:

```bash
# Make sure venv is active, then from the project root:
uvicorn src.api:app --reload --port 8000
```

Wait until you see:
```
[API] Ready to serve requests.
INFO:     Application startup complete.
```

Then run the cells below.

In [ ]:
import httpx
import json

BASE_URL = "http://localhost:8000"

# --- Health check ---
resp = httpx.get(f"{BASE_URL}/health")
print("GET /health")
print(json.dumps(resp.json(), indent=2))

In [ ]:
# --- POST /ask ---
payload = {
    "question"  : "I want to cancel my order",
    "top_k"     : 3,
    "use_hybrid": True
}
resp = httpx.post(f"{BASE_URL}/ask", json=payload, timeout=60)
data = resp.json()

print("POST /ask")
print(f"  Query  : {data['query']}")
print(f"  Answer : {data['answer']}")
print(f"  Sources: {len(data['sources'])} docs retrieved")
for src in data['sources']:
    print(f"    [{src['intent']}]  score={src['score']:.4f}")

In [ ]:
# --- GET /search (retrieval only, no generation) ---
resp = httpx.get(f"{BASE_URL}/search", params={"query": "track my delivery", "top_k": 3})
print("GET /search?query=track my delivery&top_k=3")
for doc in resp.json():
    print(f"  [{doc['intent']}]  score={doc['score']:.4f}  ->  {doc['instruction']}")

In [ ]:
# --- Swagger UI shortcut ---
# You can also test the API interactively in your browser at:
print("Interactive API docs (Swagger UI):")
print(f"  {BASE_URL}/docs")
print()
print("Raw OpenAPI schema:")
print(f"  {BASE_URL}/openapi.json")

## Step 6 — Security notes

Your Milestone 3 requirements include: *'Secure endpoints with Azure AD or API keys'*.
Since we're running locally (no Azure yet), here is how security is handled:

**Current state (local dev):**
- CORS is open (`allow_origins=["*"]`) — fine locally, must be restricted before production
- No authentication on endpoints — intentional for local testing

**When Azure is fixed — production security checklist:**

| Layer | Local (now) | Azure (later) |
|---|---|---|
| Auth | None | Azure AD OAuth2 / API Management keys |
| CORS | `*` | Lock to your support portal domain |
| Transport | HTTP | HTTPS via Azure App Service |
| Rate limiting | None | Azure API Management policies |
| Secrets | None needed | Azure Key Vault |

**Quick local API key (optional, to show in the project):**
You can add a simple API key check to `src/api.py` by adding
this header dependency to each endpoint:
```python
from fastapi.security.api_key import APIKeyHeader
api_key_header = APIKeyHeader(name="X-API-Key")

async def verify_key(key: str = Depends(api_key_header)):
    if key != os.environ.get("API_KEY", "dev-key"):
        raise HTTPException(status_code=403, detail="Invalid API key")
```
Then set `API_KEY=your-secret` as an environment variable before running uvicorn.

## Milestone 3 — Summary

| Deliverable | Status | Where |
|---|---|---|
| RAG chain (retrieve + generate) | Done | `src/rag_chain.py` |
| REST API (`/ask`, `/search`, `/health`) | Done | `src/api.py` |
| Interactive testing | Done | this notebook |
| End-to-end evaluation (BLEU, ROUGE) | Done | Step 4 |
| Security plan | Done | Step 6 |
| Azure deployment | Pending (Azure issue) | `src/api.py` is Azure App Service ready |

**Project folder structure so far:**
```
NHA-4-231/
├── src/
│   ├── __init__.py
│   ├── rag_chain.py         <- core RAG logic
│   └── api.py               <- FastAPI REST server
├── notebooks/
│   ├── Milestone_1_VSCode.ipynb
│   ├── Milestone_2_Local.ipynb
│   └── Milestone_3_Local.ipynb  <- this file
├── data/
│   ├── train_df.csv / val_df.csv / test_df.csv
│   └── faiss_index/         <- FAISS index + embeddings from M2
└── venv/
```

---
**Next -> Milestone 4:** MLflow experiment tracking, monitoring dashboard,
and automated retraining pipeline.